In [3]:
import os
import torch
import numpy as np
import random
from torch.utils.data import Dataset, Subset, DataLoader
import segmentation_models_pytorch as smp
import time

class BuildingDataset(Dataset):
    def __init__(self, image_dir='../train_data/images', mask_dir='../train_data/masks'):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        # Get all filenames and sort them for consistency
        self.filenames = sorted([f for f in os.listdir(image_dir) if f.endswith('.npy')])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        # Load image and mask
        img = np.load(os.path.join(self.image_dir, self.filenames[idx]))
        mask = np.load(os.path.join(self.mask_dir, self.filenames[idx]))
        
        # Convert to Tensors and permute to (Channels, Height, Width)
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()
        
        return img_tensor, mask_tensor

# Initialize the main dataset object
dataset = BuildingDataset()

In [4]:
# Define Geographic Split
train_prefixes = ['MussettBayouFire-03', 'MussettBayouFire-04', 'MussettBayouFire-05']
test_prefixes = ['MussettBayouFire-01', 'MussettBayouFire-02']

train_building_indices = []
train_background_indices = []
test_indices = []

for i, filename in enumerate(dataset.filenames):
    is_train = any(prefix in filename for prefix in train_prefixes)
    is_test = any(prefix in filename for prefix in test_prefixes)
    
    if is_train:
        mask = np.load(os.path.join(dataset.mask_dir, filename))
        if np.sum(mask) > 10: # Tile contains building pixels
            train_building_indices.append(i)
        else:
            train_background_indices.append(i)
    elif is_test:
        test_indices.append(i)

# Balanced Sub-sampling: Force a 70% Building / 30% Background mix
random.seed(42)
num_bg_needed = int((len(train_building_indices) / 0.7) * 0.3)
selected_bg = random.sample(train_background_indices, min(num_bg_needed, len(train_background_indices)))

final_train_indices = train_building_indices + selected_bg
random.shuffle(final_train_indices)

train_loader = DataLoader(Subset(dataset, final_train_indices), batch_size=8, shuffle=True)
test_loader = DataLoader(Subset(dataset, test_indices), batch_size=1, shuffle=False)

print(f"✅ Data Setup Complete: {len(train_loader.dataset)} Training | {len(test_loader.dataset)} Testing")

✅ Data Setup Complete: 910 Training | 257 Testing


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize U-Net with ResNet-34
model = smp.Unet(
    encoder_name="resnet34", 
    encoder_weights="imagenet", # Leverage pre-trained shape recognition
    in_channels=3, 
    classes=1, 
    activation='sigmoid'
).to(device)

# Set the loss to prioritize finding buildings (alpha=0.7)
criterion = smp.losses.TverskyLoss(mode='binary', alpha=0.7, beta=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)

# --- 30-Epoch Training Loop ---
best_loss = float('inf')

for epoch in range(30):
    model.train()
    train_loss = 0
    start = time.time()
    
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device).float()
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device).float()
            test_loss += criterion(model(imgs), masks).item()
    
    avg_test_loss = test_loss / len(test_loader)
    scheduler.step(avg_test_loss)
    
    if avg_test_loss < best_loss:
        best_loss = avg_test_loss
        torch.save(model.state_dict(), "mussett_spatial_final.pth")
        status = "🌟 Best Saved!"
    else: status = ""
        
    print(f"Epoch [{epoch+1}/30] | Train Loss: {train_loss/len(train_loader):.4f} | Test Loss: {avg_test_loss:.4f} | {status}")

Epoch [1/30] | Train Loss: 0.5564 | Test Loss: 0.6741 | 🌟 Best Saved!
Epoch [2/30] | Train Loss: 0.5333 | Test Loss: 0.6789 | 
Epoch [3/30] | Train Loss: 0.5232 | Test Loss: 0.6784 | 
Epoch [4/30] | Train Loss: 0.5158 | Test Loss: 0.6832 | 
Epoch [5/30] | Train Loss: 0.5118 | Test Loss: 0.6682 | 🌟 Best Saved!
Epoch [6/30] | Train Loss: 0.5098 | Test Loss: 0.6669 | 🌟 Best Saved!
Epoch [7/30] | Train Loss: 0.5072 | Test Loss: 0.6695 | 
Epoch [8/30] | Train Loss: 0.5042 | Test Loss: 0.6694 | 
Epoch [9/30] | Train Loss: 0.5031 | Test Loss: 0.6724 | 
Epoch [10/30] | Train Loss: 0.5014 | Test Loss: 0.6744 | 
Epoch [11/30] | Train Loss: 0.5001 | Test Loss: 0.6730 | 
Epoch [12/30] | Train Loss: 0.5001 | Test Loss: 0.6738 | 
Epoch [13/30] | Train Loss: 0.4974 | Test Loss: 0.6735 | 
Epoch [14/30] | Train Loss: 0.5004 | Test Loss: 0.6749 | 
Epoch [15/30] | Train Loss: 0.4989 | Test Loss: 0.6745 | 
Epoch [16/30] | Train Loss: 0.4969 | Test Loss: 0.6746 | 
Epoch [17/30] | Train Loss: 0.4977 | Test 

In [7]:
import torch
import numpy as np

def calculate_detailed_metrics(loader, model_path):
    # 1. Load the best saved weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    # Initialize counters for pixel-level statistics
    tp, fp, tn, fn = 0, 0, 0, 0
    
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device).float()
            
            # Get predictions and apply 0.5 threshold
            outputs = model(imgs)
            preds = (outputs > 0.5).float()
            
            # Flatten tensors for bitwise comparison
            p = preds.view(-1)
            m = masks.view(-1)
            
            # Calculate pixel-level counts
            tp += (p * m).sum().item()
            fp += (p * (1 - m)).sum().item()
            tn += ((1 - p) * (1 - m)).sum().item()
            fn += ((1 - p) * m).sum().item()

    # 2. Calculate Final Metrics
    epsilon = 1e-7 # Prevent division by zero
    
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    f1 = 2 * (precision * recall) / (precision + recall + epsilon)
    iou = tp / (tp + fp + fn + epsilon)
    specificity = tn / (tn + fp + epsilon)
    accuracy = (tp + tn) / (tp + tn + fp + fn + epsilon)
    balanced_acc = (recall + specificity) / 2

    # 3. Format output exactly as requested
    print(f"--- 📊 FINAL GEOGRAPHIC PERFORMANCE ---")
    print(f"IoU          : {iou:.4f}")
    print(f"Dice         : {f1:.4f}")
    print(f"Precision    : {precision:.4f}")
    print(f"Recall       : {recall:.4f}")
    print(f"F1-score     : {f1:.4f}")
    print(f"Specif.      : {specificity:.4f}")
    print(f"Bal. Acc     : {balanced_acc:.4f}")
    print(f"Accuracy     : {accuracy:.4f}")

# Execute the evaluation
calculate_detailed_metrics(test_loader, "mussett_spatial_final.pth")

--- 📊 FINAL GEOGRAPHIC PERFORMANCE ---
IoU          : 0.5137
Dice         : 0.6788
Precision    : 0.6125
Recall       : 0.7612
F1-score     : 0.6788
Specif.      : 0.8142
Bal. Acc     : 0.7877
Accuracy     : 0.7994


In [7]:
import torch
import numpy as np
import os
import segmentation_models_pytorch as smp

def find_best_threshold(loader, model_path):
    # 1. Setup Device internally
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 2. Define Architecture inside the function
    # We use ResNet-34 to match your training configuration.
    model = smp.Unet(
        encoder_name="resnet34", 
        in_channels=3, 
        classes=1, 
        activation='sigmoid'
    ).to(device)
    
    # 3. Load the specific weights from your path
    if not os.path.exists(model_path):
        print(f"❌ File {model_path} not found!")
        return 0.5
        
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    thresholds = np.linspace(0.1, 0.9, 17) 
    best_iou = 0
    best_threshold = 0.5

    print(f"🔍 Architecture initialized. Scanning {model_path}...")

    # 4. Extract all raw outputs for the test images
    with torch.no_grad():
        all_outputs = []
        all_masks = []
        for imgs, masks in loader:
            imgs = imgs.to(device)
            out = model(imgs).cpu().numpy()
            all_outputs.append(out)
            all_masks.append(masks.numpy())

    # 5. Iteratively find the threshold that maximizes IoU
    for t in thresholds:
        ious = []
        for out, mask in zip(all_outputs, all_masks):
            pred = (out > t).astype(float)
            
            intersection = np.sum(pred * mask)
            union = np.sum(pred) + np.sum(mask) - intersection
            
            iou = (intersection + 1e-6) / (union + 1e-6)
            ious.append(iou)
        
        mean_iou = np.mean(ious)
        if mean_iou > best_iou:
            best_iou = mean_iou
            best_threshold = t
        
        print(f"Threshold: {t:.2f} | Mean IoU: {mean_iou:.4f}")

    print(f"\n🏆 WINNER: {best_threshold:.2f} with IoU: {best_iou:.4f}")
    return best_threshold

# Execute: Just pass the loader and the filename
opt_t = find_best_threshold(test_loader, "mussett_spatial_final.pth")

🔍 Architecture initialized. Scanning mussett_spatial_final.pth...
Threshold: 0.10 | Mean IoU: 0.4011
Threshold: 0.15 | Mean IoU: 0.4006
Threshold: 0.20 | Mean IoU: 0.4001
Threshold: 0.25 | Mean IoU: 0.3998
Threshold: 0.30 | Mean IoU: 0.3997
Threshold: 0.35 | Mean IoU: 0.3995
Threshold: 0.40 | Mean IoU: 0.3993
Threshold: 0.45 | Mean IoU: 0.3990
Threshold: 0.50 | Mean IoU: 0.3987
Threshold: 0.55 | Mean IoU: 0.3984
Threshold: 0.60 | Mean IoU: 0.3981
Threshold: 0.65 | Mean IoU: 0.3978
Threshold: 0.70 | Mean IoU: 0.3974
Threshold: 0.75 | Mean IoU: 0.3975
Threshold: 0.80 | Mean IoU: 0.3971
Threshold: 0.85 | Mean IoU: 0.3958
Threshold: 0.90 | Mean IoU: 0.3935

🏆 WINNER: 0.10 with IoU: 0.4011


In [9]:
import matplotlib.pyplot as plt

def save_best_results(loader, model_path, num_images=10):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    count = 0
    with torch.no_grad():
        for i, (imgs, masks) in enumerate(loader):
            if count >= num_images: break
            
            # Skip empty tiles for the gallery
            if torch.sum(masks) < 1000: continue 
            
            imgs = imgs.to(device)
            preds = (model(imgs) > 0.5).float().cpu().numpy().squeeze()
            img_np = imgs.squeeze().permute(1,2,0).cpu().numpy()
            mask_np = masks.squeeze().cpu().numpy()
            
            # Plot and save
            fig, ax = plt.subplots(1, 3, figsize=(15, 5))
            ax[0].imshow(img_np); ax[0].set_title("Drone Image")
            ax[1].imshow(mask_np, cmap='gray'); ax[1].set_title("Ground Truth")
            ax[2].imshow(preds, cmap='jet'); ax[2].set_title("Model Clipping")
            for a in ax: a.axis('off')
            
            plt.savefig(f"best_result_{count}.png")
            plt.close()
            count += 1
    print(f"✅ Saved {count} gallery images for your report!")

save_best_results(test_loader, "mussett_spatial_final.pth")

✅ Saved 10 gallery images for your report!
